# 5.14 — Formula di Taylor

**Prerequisiti:** 5.12 Teorema di Cauchy, 4.12 Sviluppi di Taylor elementari, 5.9 Teoremi fondamentali della derivazione.


## Perché ci interessa

Nel notebook 4.12 abbiamo visto che $e^x$, $\sin x$, $\cos x$ si scrivono come serie di potenze, senza spiegare **da dove** venissero quei coefficienti (perché proprio $\frac1{n!}$?). Ora, con le derivate a disposizione, possiamo rispondere: la **formula di Taylor** costruisce il polinomio che meglio approssima una funzione vicino a un punto, usando le sue derivate successive in quel punto — e possiamo anche **misurare l'errore** di questa approssimazione, cosa che nel Modulo 4 potevamo solo osservare numericamente.

## Teoria, passo dopo passo

### Costruire l'approssimazione, un ordine alla volta

Vogliamo approssimare $f(x)$ vicino a $x_0$ con un polinomio. All'**ordine 0**, l'approssimazione più semplice è la costante $f(x_0)$ (nessuna informazione sulla pendenza). All'**ordine 1**, aggiungiamo l'informazione della derivata (notebook 5.8): la retta tangente, $f(x_0)+f'(x_0)(x-x_0)$ — un'approssimazione migliore, che "segue" la direzione di $f$ in $x_0$.

All'**ordine 2**, aggiungiamo l'informazione della derivata seconda (notebook 5.13, curvatura): vogliamo un polinomio $P_2(x)=f(x_0)+f'(x_0)(x-x_0)+a(x-x_0)^2$ che abbia anche la stessa concavità di $f$ in $x_0$. Imponendo $P_2''(x_0)=f''(x_0)$: essendo $P_2''(x)=2a$ (costante), serve $a=\frac{f''(x_0)}2$.

### Il polinomio di Taylor di ordine $n$

Generalizzando questo schema (ogni coefficiente scelto per far coincidere la derivata $k$-esima di $P_n$ con quella di $f$, in $x_0$):

$$P_n(x) = f(x_0)+f'(x_0)(x-x_0)+\frac{f''(x_0)}{2!}(x-x_0)^2+\dots+\frac{f^{(n)}(x_0)}{n!}(x-x_0)^n = \sum_{k=0}^n\frac{f^{(k)}(x_0)}{k!}(x-x_0)^k$$

Per $x_0=0$, questa si chiama anche **formula di Maclaurin**. Verifichiamo che coincida con le serie del notebook 4.12: per $f(x)=e^x$, ogni derivata è $e^x$ stessa (catalogo, notebook 5.9), quindi $f^{(k)}(0)=e^0=1$ per ogni $k$: $P_n(x)=\sum_{k=0}^n\frac{x^k}{k!}$ — esattamente i coefficienti $\frac1{n!}$ del notebook 4.12, ora spiegati.

### Il resto: quanto sbagliamo

La differenza $R_n(x)=f(x)-P_n(x)$ si chiama **resto**. Il **teorema di Taylor con resto di Lagrange** (dimostrabile applicando ripetutamente il teorema di Cauchy, notebook 5.12) afferma che esiste un punto $\xi$ tra $x_0$ e $x$ tale che:

$$R_n(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!}(x-x_0)^{n+1}$$

Questa formula generalizza il teorema di Lagrange stesso (notebook 5.11): infatti, per $n=0$, $R_0(x)=f(x)-f(x_0)=f'(\xi)(x-x_0)$ è esattamente la tesi di Lagrange!

### Perché il resto ci permette di controllare l'errore

Se riusciamo a **limitare** $|f^{(n+1)}(\xi)|$ per $\xi$ nell'intervallo tra $x_0$ e $x$ (per esempio, se sappiamo che $|f^{(n+1)}|\le M$ lì), otteniamo una stima esplicita dell'errore:

$$|R_n(x)| \le \frac{M}{(n+1)!}|x-x_0|^{n+1}$$

Per $e^x$ vicino a $0$: ogni derivata è ancora $e^x$, limitata (per esempio) da $e$ su $[0,1]$; questo permette di dire, con precisione, quanti termini servono per un'approssimazione entro una tolleranza data — la domanda lasciata aperta nel notebook 4.12, ora risolta rigorosamente.

In [ ]:
import sympy as sp

x, x0 = sp.symbols('x x0', real=True)

# Costruzione del polinomio di Taylor per e^x centrato in 0, ordine crescente
f = sp.exp(x)
for n in [1, 2, 4]:
    P_n = sum(sp.diff(f, x, k).subs(x, 0) / sp.factorial(k) * x**k for k in range(n+1))
    print(f"Polinomio di Taylor ordine {n}: {P_n}")

# Confronto con sp.series
print("\nCon sp.series (ordine 5):", sp.series(f, x, 0, 5))

# Stima dell'errore per e^0.5 con il polinomio di ordine 3
x_val = 0.5
P_3 = sum(sp.diff(f, x, k).subs(x, 0) / sp.factorial(k) * x_val**k for k in range(4))
valore_esatto = sp.exp(x_val)
errore_reale = abs(float(valore_esatto - P_3))
print(f"\nP_3(0.5) = {float(P_3):.6f}, e^0.5 = {float(valore_esatto):.6f}, errore reale = {errore_reale:.6f}")

# Stima teorica dell'errore: |R_3| <= M/4! * |x|^4, con M = max di e^xi su [0,0.5], M <= e^0.5
M = float(sp.exp(0.5))
stima_errore = M / sp.factorial(4) * x_val**4
print(f"Stima teorica dell'errore: {float(stima_errore):.6f}  (deve essere >= errore reale)")

## Esercizio svolto e commentato

**Problema:** usa la formula di Taylor con resto per stimare quanti termini della serie di $\sin x$ servono per approssimare $\sin(1)$ con un errore minore di $0.001$.

**Ragionamento:**

**Passo 1 — richiamiamo le derivate di $\sin x$** (catalogo, notebook 5.9, applicato ripetutamente): $f'=\cos x$, $f''=-\sin x$, $f'''=-\cos x$, $f^{(4)}=\sin x$, e poi il ciclo si ripete con periodo $4$. In ogni caso, **ogni** derivata di $\sin x$ è o $\pm\sin x$ o $\pm\cos x$: quindi $|f^{(k)}(\xi)|\le1$ per **ogni** $k$ e per **ogni** $\xi$ (notebook 2.9, immagine di seno e coseno).

**Passo 2 — applichiamo la stima del resto** (teoria), con $x_0=0$, $x=1$, e $M=1$ (il maggiorante appena trovato, valido per ogni ordine):

$$|R_n(1)| \le \frac{1}{(n+1)!}\times|1|^{n+1} = \frac1{(n+1)!}$$

**Passo 3 — troviamo $n$ tale che questa stima sia minore di $0.001$:**

$$\frac1{(n+1)!} < 0.001 \quad\Longleftrightarrow\quad (n+1)! > 1000$$

Calcoliamo alcuni fattoriali (notebook 4.1): $6!=720$ (non basta), $7!=5040$ (basta). Serve $n+1\ge7$, cioè $n\ge6$.

**Conclusione**: un polinomio di Taylor di ordine (almeno) $6$ approssima $\sin(1)$ con errore garantito minore di $0.001$. Poiché la serie di $\sin x$ contiene solo potenze dispari (notebook 4.12), useremo in pratica i termini fino a $x^7$ (il primo esponente dispari $\ge n=6$).

**Verifica numerica**: $P_7(1)=1-\frac16+\frac1{120}-\frac1{5040}\approx0.841468$, mentre $\sin(1)\approx0.841471$: errore reale $\approx0.000003$, ben al di sotto della soglia richiesta (la nostra stima, basata sul caso peggiore possibile per $M$, è "prudente": l'errore reale è spesso molto più piccolo).

**Perché il fatto che $|f^{(k)}|\le1$ per **ogni** ordine $k$ (una proprietà speciale di seno e coseno) rende questo calcolo dell'errore particolarmente semplice:** per altre funzioni (come $e^x$, dove le derivate crescono, notebook 3.13), il maggiorante $M$ dipenderebbe dall'intervallo scelto e tenderebbe a crescere con l'ordine, complicando la stima. Per seno e coseno, il maggiorante $M=1$ resta **sempre valido, indipendentemente dall'ordine**: una conseguenza diretta della limitatezza di queste funzioni (notebook 3.10), che rende il controllo dell'errore particolarmente pulito.

In [ ]:
import sympy as sp

x = sp.symbols('x', real=True)

# Verifica: quanti termini servono?
n = 0
fattoriale = 1
while fattoriale <= 1000:
    n += 1
    fattoriale = sp.factorial(n+1)
print(f"Serve n >= {n} (con (n+1)! = {fattoriale} > 1000)")

# Calcolo del polinomio di Taylor di sin(x) ordine 7, valutato in x=1
f = sp.sin(x)
P_7 = sum(sp.diff(f, x, k).subs(x, 0) / sp.factorial(k) * x**k for k in range(8))
P_7_val = P_7.subs(x, 1)
print(f"\nP_7(1) = {float(P_7_val):.6f}")
print(f"sin(1) = {float(sp.sin(1)):.6f}")
print(f"Errore reale: {abs(float(sp.sin(1) - P_7_val)):.8f}")

## Esercizi da risolvere

1. Scrivi il polinomio di Taylor di ordine 3 di $f(x)=\ln(1+x)$ centrato in $x_0=0$ (calcola $f',f'',f'''$ e valutale in $0$).
2. Usa il polinomio di Taylor di ordine 2 di $\cos x$ per stimare $\cos(0.2)$, confrontando con il valore reale.
3. Spiega perché, per $\cos x$, il polinomio di Taylor centrato in $0$ contiene solo potenze pari (collega alla parità, notebook 3.8, 4.12).
4. *(più impegnativo)* Stima quanti termini servono nello sviluppo di $e^x$ per approssimare $e^{0.1}$ con errore minore di $10^{-6}$ (usa che, per $x\in[0,0.1]$, ogni derivata di $e^x$ è limitata da $M=e^{0.1}<2$).

<details>
<summary><strong>👉 Soluzioni (clicca per visualizzare)</strong></summary>

**1.** $f'(x)=\frac1{1+x}$, $f''(x)=-\frac1{(1+x)^2}$, $f'''(x)=\frac2{(1+x)^3}$. In $0$: $f(0)=0$, $f'(0)=1$, $f''(0)=-1$, $f'''(0)=2$. $P_3(x)=0+x-\frac12x^2+\frac26x^3=x-\frac{x^2}2+\frac{x^3}3$.

**2.** $P_2(x)=1-\frac{x^2}2$. $P_2(0.2)=1-0.02=0.98$. Valore reale $\cos(0.2)\approx0.98007$: errore $\approx0.00007$, molto piccolo.

**3.** $\cos x$ è una funzione **pari** (notebook 3.8): $\cos(-x)=\cos(x)$. Se il polinomio di Taylor contenesse un termine con esponente dispari, il polinomio stesso non sarebbe pari (un polinomio è pari solo se ha esclusivamente esponenti pari, notebook 3.11) — contraddicendo il fatto che approssima una funzione pari. Coerentemente, le derivate dispari di $\cos x$ in $0$ sono sempre $0$ (si alternano tra $\pm\sin(0)=0$).

**4.** $|R_n(0.1)|\le\dfrac{M}{(n+1)!}(0.1)^{n+1}$ con $M<2$. Vogliamo $\dfrac{2\times(0.1)^{n+1}}{(n+1)!}<10^{-6}$. Già con $n=3$: $\dfrac{2\times0.0001}{24}\approx0.0000083$, ancora troppo grande. Con $n=4$: $\dfrac{2\times0.00001}{120}\approx0.00000017<10^{-6}$: basta $n=4$.

Verifica computazionale:
```python
import sympy as sp

x = sp.symbols('x', real=True)

f1 = sp.log(1 + x)
P1_3 = sum(sp.diff(f1, x, k).subs(x, 0) / sp.factorial(k) * x**k for k in range(4))
print("Esercizio 1:", P1_3)

f2 = sp.cos(x)
P2_2 = sum(sp.diff(f2, x, k).subs(x, 0) / sp.factorial(k) * x**k for k in range(3))
print("Esercizio 2:", P2_2.subs(x, sp.Rational(1,5)), " vs ", sp.cos(sp.Rational(1,5)).evalf())
```

</details>

## Riepilogo finale

- Il **polinomio di Taylor** $P_n(x)=\sum_{k=0}^n\frac{f^{(k)}(x_0)}{k!}(x-x_0)^k$ è costruito imponendo che le sue derivate fino all'ordine $n$ coincidano con quelle di $f$ in $x_0$.
- Il **resto di Lagrange** $R_n(x)=\frac{f^{(n+1)}(\xi)}{(n+1)!}(x-x_0)^{n+1}$ (per un opportuno $\xi$ tra $x_0$ e $x$) generalizza il teorema di Lagrange (caso $n=0$) e permette di stimare l'errore di approssimazione.
- Limitando $|f^{(n+1)}(\xi)|$ con una costante $M$, si ottiene una stima esplicita $|R_n(x)|\le\frac M{(n+1)!}|x-x_0|^{n+1}$, che permette di calcolare quanti termini servono per una precisione data.
- Le serie del notebook 4.12 (per $e^x$, $\sin x$, $\cos x$) ora hanno una spiegazione rigorosa: sono i polinomi di Taylor di ordine crescente, con coefficienti determinati dalle derivate successive nel punto $0$.

## Collegamenti

Il prossimo notebook è **5.15 — Integrale di Riemann**: dopo aver studiato a fondo la derivata, passiamo al secondo grande concetto del calcolo — l'integrale, definito come limite di somme che approssimano un'area.